# Krishi-Veda APK Builder
Run all cells. APK auto-uploads to GitHub Releases.

In [ ]:
!pip install -q buildozer cython
!sudo apt-get update -qq && sudo apt-get install -y -qq python3-pip python3-dev build-essential git autoconf automake libtool pkg-config zlib1g zlib1g-dev libncurses-dev cmake libffi-dev libssl-dev zip unzip openjdk-17-jdk

In [ ]:
import os
os.chdir('/content')
!git clone https://github.com/divineearthly/Krishi-Veda-Module.git
os.chdir('/content/Krishi-Veda-Module')

In [ ]:
from getpass import getpass
import os
token = getpass('GitHub token: ')
os.environ['GITHUB_TOKEN'] = token

In [ ]:
!export CFLAGS="-I/usr/include" && export LDFLAGS="-L/usr/lib/x86_64-linux-gnu" && yes | buildozer android debug

In [ ]:
import glob
apk = glob.glob('**/*.apk', recursive=True)
if apk:
    !cp {apk[0]} /content/krishi-veda.apk
    print('APK ready')
else:
    print('No APK found')

In [ ]:
import requests, os, json
token = os.environ.get('GITHUB_TOKEN')
if token and os.path.exists('/content/krishi-veda.apk'):
    tag = 'v2.0.0'
    # Delete old release if exists
    r = requests.get('https://api.github.com/repos/divineearthly/Krishi-Veda-Module/releases/tags/' + tag, headers={'Authorization': f'token {token}'})
    if r.status_code == 200:
        rid = r.json()['id']
        requests.delete(f'https://api.github.com/repos/divineearthly/Krishi-Veda-Module/releases/{rid}', headers={'Authorization': f'token {token}'})
    # Create new release
    r = requests.post('https://api.github.com/repos/divineearthly/Krishi-Veda-Module/releases', headers={'Authorization': f'token {token}'}, json={'tag_name': tag, 'name': 'Krishi-Veda v2.0.0', 'body': 'APK + auto-update', 'prerelease': False})
    if r.status_code == 201:
        upload_url = r.json()['upload_url'].split('{')[0]
        with open('/content/krishi-veda.apk', 'rb') as f:
            requests.post(f'{upload_url}?name=krishi-veda.apk', headers={'Authorization': f'token {token}', 'Content-Type': 'application/vnd.android.package-archive'}, data=f)
        print('Uploaded to GitHub Releases')
    else:
        print('Release failed:', r.status_code, r.text[:200])
else:
    print('No APK or token')